# 04 - LSTM 深度学习模型

**目标**: 用 BiLSTM + Attention 直接从光变曲线序列学习, 对比 XGBoost 基线。

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os

import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, accuracy_score, f1_score

%matplotlib inline

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. 加载数据并构建序列

与特征工程不同, LSTM 直接从原始光变曲线中学习, 不需要手动提取特征。
我们需要将每个天体的光变曲线转换为固定长度的序列。

In [ ]:
from src.data.download import load_plasticc
from src.data.preprocess import CLASS_NAMES

train_df, test_df, meta = load_plasticc("../data/raw/plasticc")

if meta is not None:
    # 用与 Notebook 02 相同的样本 (或重新采样)
    SAMPLES_PER_CLASS = 200
    np.random.seed(42)

    all_objects = []
    for cls_id in sorted(meta["target"].unique()):
        obj_ids = meta[meta["target"] == cls_id]["object_id"].values
        n_take = min(SAMPLES_PER_CLASS, len(obj_ids))
        chosen = np.random.choice(obj_ids, size=n_take, replace=False)
        for oid in chosen:
            all_objects.append((oid, cls_id))

    print(f"共 {len(all_objects)} 个天体")

### 构建固定长度序列

每个天体输出 shape: `(max_len, 3)` — [相对时间, 流量, 流量误差]

为简化, 我们只使用 r-band 数据。

In [ ]:
from src.data.preprocess import extract_lightcurve_by_object
from tqdm.notebook import tqdm

MAX_SEQ_LEN = 100

sequences = []
labels = []
skipped = 0

for obj_id, cls_id in tqdm(all_objects, desc="构建序列"):
    obj = train_df[train_df["object_id"] == obj_id]
    band = obj[obj["passband"] == 3]

    if len(band) < 5:
        skipped += 1
        continue

    seq = extract_lightcurve_by_object(
        pd.DataFrame(band).assign(object_id=obj_id),
        obj_id,
        max_points=MAX_SEQ_LEN
    )
    sequences.append(seq)
    labels.append(cls_id)

print(f"成功: {len(sequences)}, 跳过: {skipped}")

X_seq = np.array(sequences, dtype=np.float32)
y_seq = np.array(labels, dtype=np.int64)
print(f"X_seq: {X_seq.shape}, y_seq: {y_seq.shape}")

In [ ]:
# 标签编码
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_seq)

# 划分数据集
from sklearn.model_selection import train_test_split

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_seq, y_encoded,
    test_size=0.15, stratify=y_encoded, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp,
    test_size=0.15/0.85, stratify=y_tmp, random_state=42
)

print(f"训练: {X_train.shape}, 验证: {X_val.shape}, 测试: {X_test.shape}")
num_classes = len(label_encoder.classes_)

## 2. 创建 DataLoader

In [ ]:
BATCH_SIZE = 128

train_dataset = TensorDataset(
    torch.tensor(X_train), torch.tensor(y_train, dtype=torch.long)
)
val_dataset = TensorDataset(
    torch.tensor(X_val), torch.tensor(y_val, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test), torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 3. 训练 LSTM 模型

In [ ]:
from src.models.lstm_model import LightCurveLSTM
from src.models.train import train_model
import yaml

# 加载配置
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# 覆盖配置中的类别数
config["model"]["num_classes"] = num_classes

model = LightCurveLSTM(
    input_size=3,
    hidden_size=config["model"]["hidden_size"],
    num_layers=config["model"]["num_layers"],
    num_classes=num_classes,
    dropout=config["model"]["dropout"],
)

print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")

trained_model = train_model(
    model, train_loader, val_loader,
    config,
    save_path="../models/lstm_classifier.pt"
)

## 4. 测试集评估

In [ ]:
@torch.no_grad()
def predict_proba(model, dataloader):
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    for x_batch, y_batch in dataloader:
        x_batch = x_batch.to(DEVICE)
        logits = model(x_batch)
        probs = torch.softmax(logits, dim=1)
        all_probs.append(probs.cpu().numpy())
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_labels.append(y_batch.numpy())
    return (
        np.concatenate(all_probs),
        np.concatenate(all_preds),
        np.concatenate(all_labels)
    )

# 加载最佳模型
model = LightCurveLSTM(
    input_size=3, hidden_size=128, num_layers=2,
    num_classes=num_classes, dropout=0.3
)
model.load_state_dict(torch.load("../models/lstm_classifier.pt"))
model = model.to(DEVICE)

probs, preds, labels = predict_proba(model, test_loader)

acc = accuracy_score(labels, preds)
f1_macro = f1_score(labels, preds, average="macro")
f1_weighted = f1_score(labels, preds, average="weighted")

print(f"LSTM Test Accuracy:   {acc:.4f}")
print(f"LSTM Test Macro-F1:   {f1_macro:.4f}")
print(f"LSTM Test Weighted-F1: {f1_weighted:.4f}")

In [ ]:
class_names = [CLASS_NAMES.get(c, f"class_{c}") for c in label_encoder.classes_]
print("\nLSTM 分类报告:")
print(classification_report(labels, preds, target_names=class_names, digits=3))

## 5. XGBoost vs LSTM 对比

In [ ]:
import joblib

# 尝试加载 XGBoost 模型对比
try:
    xgb_model = joblib.load("../models/xgboost_baseline.pkl")
    # XGBoost 的特征需要从 processed 数据加载
    test_npz = np.load("../data/processed/test.npz")
    X_test_feat, y_test_feat = test_npz["X"], test_npz["y"]
    xgb_preds = xgb_model.predict(X_test_feat)
    xgb_acc = accuracy_score(y_test_feat, xgb_preds)
    xgb_f1 = f1_score(y_test_feat, xgb_preds, average="macro")

    print("模型性能对比:")
    print(f"{'':>12}  Accuracy  Macro-F1")
    print(f"{'XGBoost':>12}  {xgb_acc:.4f}    {xgb_f1:.4f}")
    print(f"{'LSTM':>12}  {acc:.4f}    {f1_macro:.4f}")
except FileNotFoundError:
    print("未找到 XGBoost 模型, 跳过对比。请先运行 03_baseline_model notebook。")

## 6. 下一步

- **05_human_review**: 找出模型不确定的样本, 人工校正并迭代优化